In [1]:
import os
import warnings
import time

import dask.dataframe as dd
import pyarrow.parquet as pq
import pyarrow as pa
from fastparquet import write

import pandas as pd
import numpy as np
import math

import pickle

from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.graph_objects as go

import lifelines
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw

import tensorflow as tf
import tensorflow_probability as tfp

config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True
sess = tf.compat.v1.Session(config = config)

from tensorflow import keras
from tensorflow.keras import optimizers, initializers, regularizers, layers

import scipy.stats as stats
from scipy.stats import norm, t, probplot, pearsonr, spearmanr, rankdata
from scipy.stats import truncnorm as truncnorm_scipy
from scipy.stats import gamma as gamma_dist
from scipy.special import gamma

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold

import thetaflow as thf

import json
import gc
import glob
from pathlib import Path

2026-05-28 05:18:14.791211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779956294.809138   48062 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779956294.814550   48062 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779956294.829015   48062 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779956294.829042   48062 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779956294.829045   48062 computation_placer.cc:177] computation placer alr

In [2]:
def survival_data_loader_generator(file_path, time_col, event_col):
    '''
        Lê um arquivo parquet em batches e cospe os tensores para o modelo.
    '''
    
    parquet_file = pq.ParquetFile(file_path)
    num_groups = parquet_file.num_row_groups
    # Itera sobre os blocos nativos do Parquet
    for i in range(num_groups):
        # Lê o i-ésimo grupo de observações separado no arquivo parquet
        table = parquet_file.read_row_group(i)
        
        # Converte apenas este batch para um DataFrame Pandas/Numpy
        df_batch = table.to_pandas()
        
        # Variáveis de Sobrevivência (Y)
        time = df_batch[time_col].values.astype(np.float32)
        event = df_batch[event_col].values.astype(np.float32)

        # Remove as colunas de tempo e censura, mantendo apenas as features
        df_batch = df_batch.drop(columns = [time_col, event_col])
        # Matriz de Features (X)
        X = df_batch.values.astype(np.float32)
        
        # As variáveis resposta são retornadas como uma lista
        # y = (time, event)
        
        # Entrega o batch pronto para o TensorFlow
        yield X, time, event

In [11]:
def survival_data_loader_generator(file_path, time_col, event_col):
    parquet_file = pq.ParquetFile(file_path)
    num_groups = parquet_file.num_row_groups
    
    for i in range(num_groups):
        table = parquet_file.read_row_group(i)
        
        # 1. Extração direta para NumPy (Muito mais rápido que .to_pandas())
        # pyarrow extrai a coluna e converte direto pro C-backend do numpy
        time = table.column(time_col).to_numpy().astype(np.float32)
        event = table.column(event_col).to_numpy().astype(np.float32)
        
        # 2. Dropamos as colunas alvo da tabela do pyarrow
        feature_cols = [c for c in table.column_names if c not in [time_col, event_col]]
        table_features = table.select(feature_cols)
        
        # 3. Conversão para NumPy 2D agilizada
        # O to_pandas() aqui é inevitável para colar as colunas colunares em matriz 2D,
        # MAS, como os dados já estão limpos e numéricos, fazemos de forma mais enxuta
        X = table_features.to_pandas().values.astype(np.float32)
        
        yield X, time, event

In [12]:
train_data_path = "train_data.parquet"
n_features = len(dd.read_parquet(train_data_path).columns)
time_col = "tempo"
event_col = "delta"
batch_size = 20000

# Criando o Dataset base com a nova Assinatura
train_dataset = tf.data.Dataset.from_generator(
    lambda: survival_data_loader_generator(train_data_path, time_col, event_col),
    
    output_signature=(
        # 1. Matriz X (Features)
        tf.TensorSpec(shape=(None, n_features-2), dtype=tf.float32),
        # 2. Vetor time (Tempo)
        tf.TensorSpec(shape=(None,), dtype=tf.float32), 
        # 3. Vetor event (Evento/Censura)
        tf.TensorSpec(shape=(None,), dtype=tf.float32)  
    )
)

# Otimização do pipeline
train_dataset = (
    train_dataset
    .unbatch()                                                                 # Quebra os 512 blocos gigantes em linhas individuais fluidas
    .shuffle(buffer_size = 40000, reshuffle_each_iteration = True, seed = 10)  # Embaralha as observações de cada batch
    .batch(batch_size)                                                         # Reagrupa em lotes exatos de batch_size para estabilizar o gradiente da Rede Neural
    .prefetch(tf.data.AUTOTUNE)                                                # Manda a CPU preparar o próximo batch enquanto a GPU treina o atual
)

In [13]:
count = 0
start = time.perf_counter()
for i, batch_data in enumerate(train_dataset):
    count += 1
    print(count)
print(time.perf_counter()-start)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
